# Corpus regression dataset

In [2]:
from __future__ import annotations

from transformers import AutoTokenizer

from src import get_repo_base
from src.data.corpus_regression import (
    CorpusRegressionDataloadingConfig,
    CorpusRegressionDatasetConfig,
    CorpusRegressionDataset,
)

In [ ]:
dataset_config = CorpusRegressionDatasetConfig.get_canonical()
out_dir = dataset_config.get_canonical_folder(get_repo_base() / "artifacts" / "corpus-regression")
display(dataset_config.visualize())

# Build-or-load
CorpusRegressionDatasetConfig.init_or_load_from(folder=out_dir, **dataset_config.model_dump())
ds = CorpusRegressionDataset.load_from(out_dir)
print(f"train_tokens: {tuple(ds.train_tokens.shape)}  dtype={ds.train_tokens.dtype}")
print(f"train_labels: {tuple(ds.train_labels.shape)}  dtype={ds.train_labels.dtype}")
print(f"val_tokens:   {tuple(ds.val_tokens.shape)}")
print(f"val_labels:   {tuple(ds.val_labels.shape)}")
print(f"label mean:   {ds.train_labels.mean().item():+.4f}  (≈0 expected)")
print(f"label std:    {ds.train_labels.std().item():+.4f}  (≈1 expected)")

### Inspect a sample

Decode a prefix back to text and print its label vector.

In [6]:
tok = AutoTokenizer.from_pretrained(dataset_config.pretrained_tokenizer_model_name)
prefix_ids = ds.train_tokens[0].tolist()
print(f"prefix ({len(prefix_ids)} tokens):")
print(tok.decode(prefix_ids)[:400] + " …")
print(f"\nlabel ({ds.train_labels.shape[1]}-dim ±1):")
print(ds.train_labels[0].tolist())

prefix (128 tokens):
In the measurement of battery technology, there are four kinds common method (open-circuit voltage measurement, Coulomb calculation, impedance measurement, integrated look-up table method), usually using a combination of methods to one of the main method of supporting the rest of the way computing power.
The first one is the open circuit voltage measurement, this method is that measuring the batte …

label (32-dim ±1):
[1.0, 1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]


## 3. Dataloaders

`CorpusRegressionDataloadingConfig` wraps each split in a `TensorDataset` and returns a `DataLoader`. Single-device by default; shuffle on for train, off for val.

In [ ]:
dl_cfg = CorpusRegressionDataloadingConfig(
    train_batch_size=256,
    eval_batch_size=512,
    drop_last=True,
)
train_dl = dl_cfg.get_train_dataloader(ds)
val_dl = dl_cfg.get_val_dataloader(ds)

batch_tokens, batch_labels = next(iter(train_dl))
print(f"batch tokens: {tuple(batch_tokens.shape)}  dtype={batch_tokens.dtype}")
print(f"batch labels: {tuple(batch_labels.shape)}  dtype={batch_labels.dtype}")
print(f"len(train_dl)={len(train_dl)}  len(val_dl)={len(val_dl)}")